# Milestone - Unit 6: evaluation and the first blend -- five retrieval paths, one calibrated weighted blend, and an honest scoreboard that asks not just *how often* we hit, but *how well* we rank, *how much* catalog we reach, and whether every path actually earns its place.

## 1. Load the catalog, interactions, keyword documents, and cold readers

The same generated substrate Units 1-5 ran on. We load the keyword bags so the Unit-3
lexical path can be scored beside the others, and the cold-reader partition so the frozen
`val` scoreboard excludes the 60 readers no personalized path can serve.
`bookrec.generated_dir()` reports a clear error if `recsys/data/generated/` has not been built.

In [ ]:
import json
from collections import defaultdict

import bookrec
import pandas as pd
from bookrec import (
    blend,
    catalog_coverage,
    hit_rate_at_k,
    intra_list_diversity,
    ndcg_at_k,
    novelty,
    precision_at_k,
    rank,
    run_blended_scoreboard,
    run_validation_scoreboard,
)

K = 10
POOL = 30
RANDOM_SEED = 0

data_dir = bookrec.generated_dir()
catalog = bookrec.load_catalog(data_dir / "catalog.csv.gz")
catalog_ids = sorted(catalog)
keywords = bookrec.load_keywords(data_dir / "keywords.csv.gz")

cold = json.loads((data_dir / "cold_partitions.json").read_text())
cold_readers = cold["cold_readers"]

interactions_path = data_dir / "interactions.csv.gz"
rows = pd.read_csv(interactions_path).to_dict("records")

{
    "books": len(catalog_ids),
    "interactions": len(rows),
    "cold_readers": len(cold_readers),
}

## 2. Fit all five Part-1 paths and register them

Five signals, each a `RetrievalPath` with the same `fit`/`retrieve` contract: the seeded
random floor, crowd popularity counts, Unit-3 lexical content words, Unit-4 item-item
co-occurrence CF, and Unit-5 matrix factorization (the one real training fit, ~15 s, at its
pinned config). Registering them in a `PathRegistry` lets the scoreboard loop over them
uniformly. The MF seed is fixed, so the whole demo is deterministic.

In [ ]:
random_floor = bookrec.RandomRetrievalPath(catalog_ids, seed=RANDOM_SEED)
popularity = bookrec.PopularityRetrievalPath().fit(rows, catalog=catalog_ids)
lexical = bookrec.LexicalRetrievalPath(keywords).fit(rows, catalog=catalog_ids)
cf = bookrec.ItemItemRetrievalPath().fit(rows, catalog=catalog_ids)
mf = bookrec.MatrixFactorizationPath(seed=RANDOM_SEED).fit(rows, catalog=catalog_ids)  # ~15 s

registry = bookrec.PathRegistry()
for path in (random_floor, popularity, lexical, cf, mf):
    registry.register(path)

[path.name for path in registry]

## 3. Build the evaluation harness: eligibility, item-item similarity, popularity counts

Every metric below scores the SAME readers the frozen `val` scoreboard scores: a reader's
`train` positives are the seen set (never recommended back), their `val` positives are the
held-out relevant set, and the 60 cold readers (plus any reader with no unseen `val` positive)
are excluded -- 500 readers remain. We reproduce that eligibility once here so the single-path
table, the blend, and the ablation all share it. Intra-list diversity needs an item-item
similarity: we use the Unit-4 CF co-occurrence matrix as the canonical one. Novelty needs each
book's train-positive share, which is exactly the popularity path's counts.

In [ ]:
known_items = set(catalog_ids)
cold_set = set(cold_readers)

seen_by_reader = defaultdict(set)
relevant_by_reader = defaultdict(set)
for row in rows:
    if row["item_id"] not in known_items or int(row["label"]) != 1:
        continue
    if row["split"] == "train":
        seen_by_reader[row["reader_id"]].add(row["item_id"])
    elif row["split"] == "val":
        relevant_by_reader[row["reader_id"]].add(row["item_id"])

eligible_readers = [
    reader_id
    for reader_id in sorted(relevant_by_reader)
    if reader_id not in cold_set
    and (relevant_by_reader[reader_id] - seen_by_reader[reader_id])
]

# Intra-list diversity similarity: the Unit-4 CF item-item cosine matrix, addressed by item id.
cf_state = cf.artifact()
cf_items = cf_state["item_ids"]
cf_similarity = cf_state["similarity"]
cf_row_of = {item_id: index for index, item_id in enumerate(cf_items)}

def item_similarity(a, b):
    ia = cf_row_of.get(a)
    ib = cf_row_of.get(b)
    if ia is None or ib is None:
        return 0.0
    return float(cf_similarity[ia, ib])

# Novelty popularity: each book's train-positive count (a rarer book is more novel).
train_counts = popularity.counts

def recommend_all(recommend):
    """Run a (reader_id, seen) -> recommendations function over every eligible reader."""
    rec_lists = []
    relevant_lists = []
    for reader_id in eligible_readers:
        seen = seen_by_reader[reader_id]
        relevant = relevant_by_reader[reader_id] - seen
        rec_lists.append(recommend(reader_id, seen))
        relevant_lists.append(relevant)
    return rec_lists, relevant_lists

def mean_metric(metric, rec_lists, relevant_lists):
    total = sum(metric(recs, rel, K) for recs, rel in zip(rec_lists, relevant_lists))
    return total / len(rec_lists)

def beyond_accuracy(rec_lists):
    coverage = catalog_coverage(rec_lists, catalog_ids)
    diversity = sum(intra_list_diversity(recs, item_similarity) for recs in rec_lists) / len(rec_lists)
    mean_novelty = sum(novelty(recs, train_counts) for recs in rec_lists) / len(rec_lists)
    return coverage, diversity, mean_novelty

{"eligible_readers": len(eligible_readers)}

## 4. The deepened single-path scoreboard on `val`

Hit@k only asks whether a relevant book appears anywhere in the top 10. **Precision@k** asks
how many of the 10 slots are relevant, and **NDCG@k** rewards putting the right book *higher*
(a hit at rank 1 is worth more than one at rank 10). Beyond accuracy, **coverage** is the share
of the 2000-book catalog that ever surfaces, **diversity** is 1 - mean pairwise similarity
within a reader's list, and **novelty** is how rare the recommended books are.
`run_validation_scoreboard` gives the ranking metrics; we add the beyond-accuracy columns from
the shared harness and assert the harness hit matches the scoreboard's.

In [ ]:
single_rows = []
for path in registry:
    board = run_validation_scoreboard(
        path, interactions_path, catalog_ids=catalog_ids, k=K, cold_readers=cold_readers
    )
    rec_lists, relevant_lists = recommend_all(
        lambda reader_id, seen, path=path: rank(
            path.retrieve(reader_id, {"seen": seen}, K), n=K, exclude=seen
        )
    )
    hit = mean_metric(hit_rate_at_k, rec_lists, relevant_lists)
    precision = mean_metric(precision_at_k, rec_lists, relevant_lists)
    dcg = mean_metric(ndcg_at_k, rec_lists, relevant_lists)
    # Deterministic paths must reproduce the official scoreboard exactly (binding the
    # beyond-accuracy columns to shipped code); the random floor is stochastic and stateful,
    # so its RNG has already advanced and it is exempt from the equality check.
    if path.name != "random":
        assert abs(hit - board.hit_rate_at_k) < 1e-9, path.name
        assert abs(precision - board.precision_at_k) < 1e-9, path.name
        assert abs(dcg - board.ndcg_at_k) < 1e-9, path.name
    coverage, diversity, mean_novelty = beyond_accuracy(rec_lists)
    single_rows.append({
        "path": path.name,
        "hit@10": round(hit, 4),
        "precision@10": round(precision, 4),
        "ndcg@10": round(dcg, 4),
        "coverage": round(coverage, 4),
        "diversity": round(diversity, 4),
        "novelty": round(mean_novelty, 4),
    })

scoreboard = {entry["path"]: entry for entry in single_rows}

# These single-path numbers are the ground truth the lesson quotes (seed 0, k=10, 500 val readers).
assert abs(scoreboard["popularity"]["hit@10"] - 0.108) < 0.001
assert abs(scoreboard["lexical"]["hit@10"] - 0.158) < 0.001
assert abs(scoreboard["item-item"]["hit@10"] - 0.252) < 0.001
assert abs(scoreboard["mf"]["hit@10"] - 0.276) < 0.001
assert abs(scoreboard["popularity"]["coverage"] - 0.008) < 0.001
assert abs(scoreboard["item-item"]["coverage"] - 0.490) < 0.001
assert abs(scoreboard["mf"]["coverage"] - 0.0745) < 0.001

pd.DataFrame(single_rows)

Read it honestly. MF has the best ranking metrics (hit@10 ~0.276, the top precision and NDCG)
but reaches only ~7% of the catalog. Item-item CF is close on accuracy (~0.252) and reaches
~49%. Popularity is weak (~0.108) and nearly flat (0.8% coverage -- the same head books for
everyone), though its recommendations are the least novel, as expected. Lexical sits between.
No single path is best at everything: the path that ranks best serves the smallest slice of the
catalog. That tension is exactly what a blend is for.

## 5. The first blended recommender -- calibrate, weight, union, rank

Blending calibrates each path's scores to a common [0, 1] scale, takes the weighted union of
their candidate pools, and orders the result with the classical ranker (dropping each reader's
seen books). The config is **pinned** -- per-path retrieval pool 30, weights
{popularity 0.25, lexical 0.5, item-item 1.0, mf 1.0} -- chosen on `val` and frozen here, NOT
tuned against the scoreboard we are about to read (that would be the anti-lesson). We score the
blend with `run_blended_scoreboard` for the official hit@10 and coverage, then add the ranking
and beyond-accuracy columns from the shared harness.

In [ ]:
blend_paths = {"popularity": popularity, "lexical": lexical, "item-item": cf, "mf": mf}
blend_weights = {"popularity": 0.25, "lexical": 0.5, "item-item": 1.0, "mf": 1.0}

blend_board = run_blended_scoreboard(
    blend_paths, interactions_path, catalog_ids=catalog_ids, k=K, pool=POOL,
    weights=blend_weights, cold_readers=cold_readers,
)

def blended_recs(reader_id, seen):
    per_path = {
        name: path.retrieve(reader_id, {"seen": seen}, POOL)
        for name, path in blend_paths.items()
    }
    return rank(blend(per_path, weights=blend_weights), n=K, exclude=seen)

blend_recs, blend_relevant = recommend_all(blended_recs)
harness_blend_hit = mean_metric(hit_rate_at_k, blend_recs, blend_relevant)
blend_precision = mean_metric(precision_at_k, blend_recs, blend_relevant)
blend_ndcg = mean_metric(ndcg_at_k, blend_recs, blend_relevant)
blend_coverage, blend_diversity, blend_novelty = beyond_accuracy(blend_recs)

# The harness reproduces the official scoreboard, so the extra columns are bound to shipped code.
assert abs(harness_blend_hit - blend_board.hit_rate_at_k) < 1e-9
assert abs(blend_coverage - blend_board.catalog_coverage) < 1e-9

mf_hit = scoreboard["mf"]["hit@10"]
mf_coverage = scoreboard["mf"]["coverage"]
# The two binding facts: the blend beats the best single path AND covers far more catalog.
assert blend_board.hit_rate_at_k >= mf_hit - 0.01
assert blend_board.catalog_coverage >= 3 * mf_coverage
assert abs(blend_board.hit_rate_at_k - 0.306) < 0.001
assert abs(blend_board.catalog_coverage - 0.333) < 0.001

full_board = single_rows + [{
    "path": "BLEND",
    "hit@10": round(blend_board.hit_rate_at_k, 4),
    "precision@10": round(blend_precision, 4),
    "ndcg@10": round(blend_ndcg, 4),
    "coverage": round(blend_board.catalog_coverage, 4),
    "diversity": round(blend_diversity, 4),
    "novelty": round(blend_novelty, 4),
}]
pd.DataFrame(full_board)

The blend lands at **hit@10 ~0.306** -- above MF's 0.276, the best single path -- while
reaching **~33% of the catalog**, roughly 4.5x MF's 7%. Its precision and NDCG also edge past
every single path. So the honest headline holds: a calibrated, sensibly-weighted blend beats
the best single path AND serves far more of the catalog. But 'it beats the best path' is a
measured result at this pinned config, not a law -- the next cell shows a blend can also carry
dead weight.

## 6. Leave-one-path-out ablation -- does every path earn its place?

Drop one path, re-blend the other three (same pool, same weights), and measure the signed
change in hit@10. A path that genuinely contributes should make hit@10 *fall* when it is
removed.

In [ ]:
ablation_rows = []
for dropped in blend_paths:
    kept = {name: path for name, path in blend_paths.items() if name != dropped}
    kept_weights = {name: blend_weights[name] for name in kept}
    board = run_blended_scoreboard(
        kept, interactions_path, catalog_ids=catalog_ids, k=K, pool=POOL,
        weights=kept_weights, cold_readers=cold_readers,
    )
    ablation_rows.append({
        "dropped": dropped,
        "blend_hit@10": round(board.hit_rate_at_k, 4),
        "delta_vs_full": round(board.hit_rate_at_k - blend_board.hit_rate_at_k, 4),
    })

ablation = {entry["dropped"]: entry for entry in ablation_rows}
# Dropping popularity does NOT lower hit@10; the three reader-dependent paths each help.
assert ablation["popularity"]["delta_vs_full"] >= 0.0
assert ablation["lexical"]["delta_vs_full"] < 0.0
assert ablation["item-item"]["delta_vs_full"] < 0.0
assert ablation["mf"]["delta_vs_full"] < 0.0
assert abs(ablation["popularity"]["blend_hit@10"] - 0.308) < 0.001
assert abs(ablation["mf"]["blend_hit@10"] - 0.260) < 0.001

pd.DataFrame(ablation_rows)

**The honest headline.** Dropping lexical, item-item, or MF each *lowers* hit@10 (-0.014,
-0.026, -0.046) -- those three reader-dependent paths genuinely contribute. But dropping
**popularity** leaves hit@10 unchanged (0.306 -> 0.308, +0.002 ~ 0): its marginal accuracy
value is essentially zero. So 'every path contributes' is **false** here. The mechanism is
calibration: min-max maps every path's top item to a score of 1.0, so a reader-independent
popularity path ties its head books with every reader's true top pick -- it adds head coverage,
not accuracy. We keep popularity (down-weighted to 0.25) for exactly that head coverage, not
for hit@10.

**Why keep the blend at all, then?** Coverage and robustness. The blend reaches ~4.5x the
catalog MF alone does, and it degrades gracefully: the 60 cold readers have no `train` history,
so MF, CF, and lexical return `[]` for them -- only popularity (and therefore the blend) can
serve them at all. The accuracy win here is real but modest; a learned reranker (Unit 11) is
what turns a calibrated pool into a larger accuracy gain. Part 1 ends with a system that is
more than hit@10: it ranks, it reaches, and it keeps serving every reader.